In [ ]:
import cv2
import numpy as np
import mediapipe as mp
import speech_recognition as sr




In [2]:
mpHands=mp.solutions.hands
mpDraw=mp.solutions.drawing_utils

In [3]:
hands=mpHands.Hands(max_num_hands=1,static_image_mode=False,min_detection_confidence=0.7,min_tracking_confidence=0.7)

In [4]:
import time

In [5]:
import math

In [7]:
import winsound

In [8]:
import threading

In [10]:
video = cv2.VideoCapture(0)
status=""
speechResult= ""
isListening=False

video.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
video.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
print(video.get(cv2.CAP_PROP_FRAME_WIDTH))
print(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
prev_time = 0

class Button():
    def __init__(self, pos, text, size=(60,60)):
        self.pos = pos
        self.text = text
        self.size = size

keys = [
['Q','W','E','R','T','Y','U','I','O','P'],
['A','S','D','F','G','H','J','K','L'],
['Z','X','C','V','B','N','M']
]

buttonList = []
startX = [30,60,105]

for i,row in enumerate(keys):
    for j,key in enumerate(row):
        buttonList.append(Button((startX[i] + j*68,120 + i*70),key))

buttonList.append(Button((40,340),'SPACE',(260,60)))
buttonList.append(Button((330,340),'BACK',(150,60)))
buttonList.append(Button((690,340),'CLEAR',(170,60)))
buttonList.append(Button((510,340),'MIC',(150,60)))



def drawKeyboard(img):

    for button in buttonList:

        x,y = button.pos
        w,h = button.size

        cv2.rectangle(img,(x,y),(x+w,y+h),(255,0,255),2)

        cv2.putText(
            img,
            button.text,
            (x+15,y+38),
            cv2.FONT_HERSHEY_COMPLEX,
            0.8,
            (255,255,255),
            2
        )





finalText=""
lastClick=0



def playClick():
        winsound.Beep(700,40)


status = ""
def voiceTyping():

    global status
    global speechResult
    global isListening


    recognizer = sr.Recognizer()

    try:

        status = "Listening..."

        with sr.Microphone() as source:

            recognizer.adjust_for_ambient_noise(source, duration=0.5)

            audio = recognizer.listen(source, timeout=5)

        status = "Recognizing..."

        text = recognizer.recognize_google(audio)

        speechResult = text.upper()

        status = ""

        isListening = False

    except:

    

        speechResult = ""
        status = ""

        isListening = False
        


while True:

    success,img=video.read()

    if not success:
        break

    img=cv2.flip(img,1)

    rgb=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)

    result=hands.process(rgb)

    drawKeyboard(img)

    

    if result.multi_hand_landmarks:

        for handLms in result.multi_hand_landmarks:

            mpDraw.draw_landmarks(img,handLms,mpHands.HAND_CONNECTIONS)

            h,w,c=img.shape

            landmarks=[]

            for id,lm in enumerate(handLms.landmark):

                cx=int(lm.x*w)
                cy=int(lm.y*h)

                landmarks.append((cx,cy))

            if len(landmarks):

                x1,y1=landmarks[8]
                x2,y2=landmarks[4]

                distance=math.hypot(x2-x1,y2-y1)

                cv2.circle(img,(x1,y1),9,(255,0,255),cv2.FILLED)

                for button in buttonList:

                    x,y=button.pos
                    bw,bh=button.size

                    if x<x1<x+bw and y<y1<y+bh:

                        cv2.rectangle(img,(x,y),(x+bw,y+bh),(0,255,0),cv2.FILLED)

                        cv2.putText(
                            img,
                            button.text,
                            (x+15,y+45),
                            cv2.FONT_HERSHEY_COMPLEX,
                            0.8,
                            (0,0,0),
                            2
                        )

                        currentTime=time.time()

                        if distance<45:

                            if currentTime-lastClick>0.35:

                                if button.text=="SPACE":

                                    finalText+=" "

                                elif button.text=="BACK":

                                    finalText=finalText[:-1]

                                elif button.text=="CLEAR":
                                    finalText = ""
                                    
                                    

                                elif button.text=="MIC":
                                     if not isListening:
                                         isListening = True
                                         threading.Thread(target=voiceTyping,daemon=True).start()
                                     

                                else:

                                    finalText+=button.text

                                lastClick=currentTime
                                lastKeyPress = currentTime
                                playClick()

    currentWord = finalText.split()[-1] if finalText.strip() else ""

    if speechResult != "":
        if len(finalText) > 0 and finalText[-1] != " ":
            finalText += " "
        finalText += speechResult + " "
        speechResult = ""
        

    

    current_time=time.time()

    cv2.rectangle(img,(40,20),(1240,90),(50,50,50),cv2.FILLED)

    cv2.putText(
        img,
        finalText,
        (60,70),
        cv2.FONT_HERSHEY_COMPLEX,
        1.3,
        (0,255,255),
        3
    )

    

    fps=1/(current_time-prev_time)
    prev_time=current_time

    cv2.putText(
        img,
        f"FPS : {int(fps)}",
        (20,40),
        cv2.FONT_HERSHEY_COMPLEX,
        1,
        (0,255,0),
        2
    )

    cv2.putText(
    img,
    status,
    (60,110),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.8,
    (0,255,0),
    2
)

    cv2.imshow("Virtual Keyboard",img)

    if cv2.waitKey(1) & 0xFF==ord('q'):
        break

video.release()
cv2.destroyAllWindows()

1280.0
720.0


d:\DOWNLOADS\DL_DEC\Invisible_Keyboard\venv\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
